# Edge Temporal Smoothing and Event-Level Alerts

This notebook matches `ai-model/scripts/edge_temporal_smoothing.py`: it runs zone-based alert smoothing with `rule_engine.get_person_zones()`, `rule_engine.check_ppe_violation()`, and `rule_engine.classify_alert()`.

When the model, videos, `rule_engine.py`, or `zones.json` are not available in Kaggle, it falls back to a clearly labeled synthetic zone-alert sequence so the metric pipeline and CSV schema can still be verified.


## 1. Controls

Attach the edge pipeline folder as a Kaggle dataset input so the notebook can find `rule_engine.py` and `configs/zones.json`. You can also override paths manually below.


In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
RESULTS_DIR = KAGGLE_WORKING / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'model_candidates': [],
    'videos': [],
    'zones_path': None,
    'output_dir': RESULTS_DIR,
    'summary_csv': RESULTS_DIR / 'event_vs_frame_metrics.csv',
    'conf_threshold': 0.4,
    'iou_threshold': 0.45,
    'device': 'cuda:0',
    'smoothing_frames': [3, 5, 10],
}

# Optional manual overrides for Kaggle inputs.
MODEL_ONNX_PATH = None
EDGE_PIPELINE_DIR = None
ZONES_JSON_PATH = None
VIDEO_PATHS = None

FAST_DEBUG = True
SEED = 42
VIDEO_NAMES = ['demo_video3.mp4', 'demo_video4.mp4']
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}


## 2. Environment and Input Discovery

This cell follows the script's dependency/import pattern, then discovers the model, videos, `edge-pipeline/`, and `zones.json` from Kaggle inputs.


In [ ]:
import importlib
import json
import math
import platform
import random
import sys
from collections import Counter

import numpy as np
import pandas as pd


def version_of(module_name):
    try:
        module = importlib.import_module(module_name)
        return str(getattr(module, '__version__', 'installed'))
    except Exception:
        return 'not installed'


print('Environment')
print(f'  Python: {platform.python_version()} ({sys.executable})')
print(f'  Platform: {platform.platform()}')
print(f'  opencv-python: {version_of("cv2")}')
print(f'  numpy: {version_of("numpy")}')
print(f'  pandas: {version_of("pandas")}')
print(f'  ultralytics: {version_of("ultralytics")}')

try:
    import torch
    print(f'  torch CUDA available: {torch.cuda.is_available()}')
except Exception as exc:
    print(f'  torch CUDA available: unknown ({exc})')

try:
    import onnxruntime as ort
    providers = ort.get_available_providers()
    gpu_note = 'GPU provider available' if any('CUDA' in p for p in providers) else 'CPU only'
    print(f'  onnxruntime: {ort.__version__}')
    print(f'  ONNXRuntime providers: {providers} ({gpu_note})')
except Exception as exc:
    print('  onnxruntime: unavailable')
    print(f'  ONNXRuntime providers: unavailable ({exc})')


def first_existing(paths):
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


def find_files_by_name(names, roots=(KAGGLE_INPUT, KAGGLE_WORKING)):
    found = {name: [] for name in names}
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if path.name in found:
                found[path.name].append(path)
    return found


def find_edge_pipeline_dir():
    if EDGE_PIPELINE_DIR is not None:
        path = Path(EDGE_PIPELINE_DIR)
        return path if (path / 'rule_engine.py').exists() else None

    candidates = []
    for root in [KAGGLE_INPUT, KAGGLE_WORKING]:
        if not root.exists():
            continue
        candidates.extend(root.rglob('rule_engine.py'))

    for rule_engine_path in candidates:
        parent = rule_engine_path.parent
        if (parent / 'configs' / 'zones.json').exists():
            return parent
    return candidates[0].parent if candidates else None


def find_model_path():
    manual = first_existing([MODEL_ONNX_PATH])
    if manual is not None:
        return manual

    names = ['baseline_yolo26n_best.onnx', 'best.onnx']
    found = find_files_by_name(names)
    for name in names:
        if found[name]:
            return sorted(found[name])[0]
    return None


def find_video_paths():
    if VIDEO_PATHS is not None:
        return [Path(path) for path in VIDEO_PATHS if Path(path).exists()]

    found = find_files_by_name(VIDEO_NAMES)
    videos = []
    for name in VIDEO_NAMES:
        if found[name]:
            videos.append(sorted(found[name])[0])
    return videos


def find_zones_path(edge_dir):
    manual = first_existing([ZONES_JSON_PATH])
    if manual is not None:
        return manual
    if edge_dir is not None:
        candidate = edge_dir / 'configs' / 'zones.json'
        if candidate.exists():
            return candidate
    found = find_files_by_name(['zones.json'])
    return sorted(found['zones.json'])[0] if found['zones.json'] else None


cv2_module = None
YOLO = None
rule_engine = None
missing_dependencies = []

try:
    cv2_module = importlib.import_module('cv2')
except Exception:
    missing_dependencies.append('opencv-python')

try:
    YOLO = importlib.import_module('ultralytics').YOLO
except Exception:
    missing_dependencies.append('ultralytics')

edge_dir = find_edge_pipeline_dir()
if edge_dir is not None and str(edge_dir) not in sys.path:
    sys.path.insert(0, str(edge_dir))

try:
    rule_engine = importlib.import_module('rule_engine')
except Exception as exc:
    print(f'  rule_engine import: unavailable ({exc})')

model_path = find_model_path()
video_paths = find_video_paths()
zones_path = find_zones_path(edge_dir)

CONFIG['model_candidates'] = [model_path] if model_path else []
CONFIG['videos'] = video_paths
CONFIG['zones_path'] = zones_path

print('\nDiscovered Inputs')
print('  edge-pipeline dir:', edge_dir)
print('  rule_engine:', 'available' if rule_engine is not None else 'unavailable')
print('  model:', model_path)
print('  videos:', [p.name for p in video_paths])
print('  zones.json:', zones_path)
print('  missing dependencies:', missing_dependencies if missing_dependencies else 'none')


## 3. Zone-Based Alert and Smoothing Functions

These functions mirror the standalone script. Real video inference uses zone keys, not `person_id`, and event counts come from the consecutive zone-alert state machine.


In [ ]:
ALERT_PRIORITY = {'NORMAL': 0, 'WARNING': 1, 'CRITICAL': 2}


def load_zones():
    with open(CONFIG['zones_path'], 'r', encoding='utf-8') as f:
        return json.load(f)


def highest_alert(levels):
    if not levels:
        return 'NORMAL'
    return max(levels, key=lambda level: ALERT_PRIORITY.get(level, 0))


def update_zone_alert(current_alerts, zone_key, alert_level):
    existing_level = current_alerts.get(zone_key, 'NORMAL')
    if ALERT_PRIORITY[alert_level] > ALERT_PRIORITY.get(existing_level, 0):
        current_alerts[zone_key] = alert_level


def detections_from_results(results, model):
    persons = []
    other_detections = []

    for result in results:
        names = getattr(result, 'names', None) or getattr(model, 'names', {})
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = math.ceil(float(box.conf[0]) * 100) / 100
            cls_idx = int(box.cls[0])
            cls_name = names.get(cls_idx, str(cls_idx)) if isinstance(names, dict) else str(cls_idx)
            det_obj = {
                'box': (x1, y1, x2, y2),
                'conf': conf,
                'class_name': cls_name,
            }
            if cls_name.lower() == 'person':
                persons.append(det_obj)
            else:
                other_detections.append(det_obj)

    return persons, other_detections


def frame_alerts_from_detections(persons, other_detections, zones_config, rule_engine):
    current_alerts = {}

    for person in persons:
        x1, _y1, x2, y2 = person['box']
        center_point = ((x1 + x2) // 2, y2)
        active_zones = rule_engine.get_person_zones(center_point, zones_config)
        ppe_violations = rule_engine.check_ppe_violation(person['box'], other_detections)
        alert_level = rule_engine.classify_alert(active_zones, ppe_violations)

        if alert_level != 'NORMAL':
            zone_key = '|'.join([z['id'] for z in active_zones]) if active_zones else 'NO_ZONE'
            update_zone_alert(current_alerts, zone_key, alert_level)

    violation_classes = {
        'no_helmet',
        'no_goggle',
        'no_gloves',
        'no_boots',
        'no_vest',
        'none',
    }

    for det in other_detections:
        vx_center = (det['box'][0] + det['box'][2]) // 2
        vy_center = (det['box'][1] + det['box'][3]) // 2
        inside_person = False

        for person in persons:
            px1, py1, px2, py2 = person['box']
            if px1 <= vx_center <= px2 and py1 <= vy_center <= py2:
                inside_person = True
                break

        if inside_person:
            continue

        center_point = (vx_center, det['box'][3])
        active_zones = rule_engine.get_person_zones(center_point, zones_config)
        cls_name_lower = det['class_name'].lower()
        ppe_violations = [det['class_name']] if cls_name_lower in violation_classes else []
        alert_level = rule_engine.classify_alert(active_zones, ppe_violations)

        if alert_level != 'NORMAL':
            zone_key = '|'.join([z['id'] for z in active_zones]) if active_zones else 'NO_ZONE'
            update_zone_alert(current_alerts, zone_key, alert_level)

    return current_alerts


def count_smoothed_events(zone_alerts_by_frame, required_frames):
    states = {}
    event_counts = Counter()

    for current_alerts in zone_alerts_by_frame:
        active_zone_keys = set(current_alerts.keys())
        tracked_zone_keys = set(states.keys()).union(active_zone_keys)

        for zone_key in tracked_zone_keys:
            current_level = current_alerts.get(zone_key, 'NORMAL')
            state = states.setdefault(
                zone_key,
                {'level': 'NORMAL', 'count': 0, 'confirmed_level': 'NORMAL'},
            )

            if current_level == state['level']:
                state['count'] += 1
            else:
                state['level'] = current_level
                state['count'] = 1
                state['confirmed_level'] = 'NORMAL'

            if current_level in {'WARNING', 'CRITICAL'} and state['count'] == required_frames:
                event_counts[current_level] += 1
                state['confirmed_level'] = current_level

    return event_counts


def process_video(video_path, model, zones_config, rule_engine, cv2_module):
    cap = cv2_module.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')

    frame_counts = Counter()
    zone_alerts_by_frame = []
    total_frames = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        total_frames += 1
        results = model.predict(
            frame,
            verbose=False,
            conf=CONFIG['conf_threshold'],
            iou=CONFIG['iou_threshold'],
            device=CONFIG['device'],
        )
        persons, other_detections = detections_from_results(results, model)
        zone_alerts = frame_alerts_from_detections(
            persons,
            other_detections,
            zones_config,
            rule_engine,
        )
        frame_level = highest_alert(zone_alerts.values())
        frame_counts[frame_level] += 1
        zone_alerts_by_frame.append(zone_alerts)

    cap.release()
    return {
        'total_frames': total_frames,
        'frame_counts': frame_counts,
        'zone_alerts_by_frame': zone_alerts_by_frame,
    }


def build_rows(video_path, metrics):
    video_name = video_path.name if hasattr(video_path, 'name') else str(video_path)
    total_frames = metrics['total_frames']
    frame_counts = metrics['frame_counts']
    frame_alert_count = frame_counts['WARNING'] + frame_counts['CRITICAL']
    rows = [
        {
            'video': video_name,
            'level': 'FRAME_LEVEL',
            'N_frames': 0,
            'total_frames': total_frames,
            'warning_count': frame_counts['WARNING'],
            'critical_count': frame_counts['CRITICAL'],
            'alert_rate': (frame_alert_count / total_frames) if total_frames else 0.0,
            'fp_reduction_pct': 0.0,
        }
    ]

    for n_frames in CONFIG['smoothing_frames']:
        event_counts = count_smoothed_events(metrics['zone_alerts_by_frame'], n_frames)
        event_count = event_counts['WARNING'] + event_counts['CRITICAL']
        if frame_alert_count > 0:
            reduction = max(0.0, (1.0 - (event_count / frame_alert_count)) * 100.0)
        else:
            reduction = 0.0
        rows.append(
            {
                'video': video_name,
                'level': 'EVENT_LEVEL',
                'N_frames': n_frames,
                'total_frames': total_frames,
                'warning_count': event_counts['WARNING'],
                'critical_count': event_counts['CRITICAL'],
                'alert_rate': (event_count / total_frames) if total_frames else 0.0,
                'fp_reduction_pct': reduction,
            }
        )

    return rows


## 4. Synthetic Fallback

This fallback is only used when the model, demo videos, `rule_engine.py`, `zones.json`, or required runtime packages are unavailable. It is synthetic and should not be interpreted as video performance.


In [ ]:
def make_synthetic_metrics():
    zone_alerts_by_frame = []
    frame_counts = Counter()

    for frame_idx in range(80):
        current_alerts = {}
        if 10 <= frame_idx <= 13:
            current_alerts['Z01'] = 'WARNING'
        if 20 <= frame_idx <= 34:
            current_alerts['Z02'] = 'CRITICAL'
        if frame_idx in {45, 47, 49}:
            current_alerts['NO_ZONE'] = 'WARNING'
        if 55 <= frame_idx <= 62:
            current_alerts['Z01'] = 'WARNING'

        frame_level = highest_alert(current_alerts.values())
        frame_counts[frame_level] += 1
        zone_alerts_by_frame.append(current_alerts)

    return {
        'total_frames': len(zone_alerts_by_frame),
        'frame_counts': frame_counts,
        'zone_alerts_by_frame': zone_alerts_by_frame,
    }


## 5. Run Zone-Based Temporal Smoothing

Real-video rows match the standalone script's output schema. The notebook uses `display()` tables instead of the script's formatted stdout tables.


In [ ]:
required_real_inputs = {
    'model': model_path is not None,
    'two_demo_videos': len(video_paths) == len(VIDEO_NAMES),
    'zones_json': zones_path is not None,
    'rule_engine': rule_engine is not None,
    'cv2': cv2_module is not None,
    'ultralytics': YOLO is not None,
}

use_synthetic = not all(required_real_inputs.values())

all_rows = []
video_metrics = []

if use_synthetic:
    print('SYNTHETIC FALLBACK ACTIVE - real video inference was not run.')
    print('Unavailable real-input checks:', {k: v for k, v in required_real_inputs.items() if not v})
    metrics = make_synthetic_metrics()
    video_name = 'synthetic_zone_alert_sequence'
    video_metrics.append((video_name, metrics))
    all_rows.extend(build_rows(video_name, metrics))
else:
    print('Pseudo-stability metrics only: no ground-truth annotations are used.')
    print(f'Selected model: {model_path}')
    print(f'Zones: {CONFIG["zones_path"]}')

    zones_config = load_zones()
    model = YOLO(str(model_path))

    for video_path in CONFIG['videos']:
        print(f'Processing {video_path.name}')
        metrics = process_video(video_path, model, zones_config, rule_engine, cv2_module)
        video_metrics.append((video_path, metrics))
        all_rows.extend(build_rows(video_path, metrics))

metrics_df = pd.DataFrame(
    all_rows,
    columns=[
        'video',
        'level',
        'N_frames',
        'total_frames',
        'warning_count',
        'critical_count',
        'alert_rate',
        'fp_reduction_pct',
    ],
)

frame_count_rows = []
for video_path, metrics in video_metrics:
    video_name = video_path.name if hasattr(video_path, 'name') else str(video_path)
    counts = metrics['frame_counts']
    frame_count_rows.append(
        {
            'video': video_name,
            'normal_count': counts['NORMAL'],
            'warning_count': counts['WARNING'],
            'critical_count': counts['CRITICAL'],
            'total_frames': metrics['total_frames'],
        }
    )

frame_counts_df = pd.DataFrame(frame_count_rows)
display(frame_counts_df)
display(metrics_df)


## 6. Save Event-vs-Frame Metrics

The output CSV is written to `/kaggle/working/results/event_vs_frame_metrics.csv` with the same schema as `ai-model/scripts/edge_temporal_smoothing.py`.


In [ ]:
out_csv = CONFIG['summary_csv']
metrics_df.to_csv(out_csv, index=False)
print('DONE - results saved to', out_csv)
print('CSV columns:', list(metrics_df.columns))
